# 008 — Homoglyph Model Evaluation

Evaluates the homoglyph-augmented Qwen LoRA adapter on PAN2025, HC3, and RAID.
Compares against the experiment 006 baseline (no augmentation).

**Requirements**: GPU runtime (T4 or better).

In [1]:
# =====================================================
# CELL 0 — Environment setup
# =====================================================
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA L4


In [2]:
# =====================================================
# CELL 1 — Install dependencies
# =====================================================
%pip install -q peft gdown scikit-learn 'datasets<3.0'

In [3]:
# =====================================================
# CELL 2 — Download datasets
# =====================================================
import json, random, requests
import gdown
from pathlib import Path
from datasets import load_dataset

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
random.seed(42)

def save_jsonl(records, path):
    with open(path, 'w') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')
    print(f'  Saved {len(records)} records to {path.name}')

# --- PAN2025 Validation ---
print('Downloading PAN2025 val...')
val_id = '12szC1TcNPN9KULPNjTZsEyG8RafxfsJy'
gdown.download(
    f'https://drive.google.com/uc?id={val_id}',
    str(DATA_DIR / 'val.jsonl'), quiet=True
)
print('  Done.')

# --- HC3 Wiki ---
print('\nDownloading HC3 (wiki subset)...')
hc3_url = 'https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main/wiki_csai.jsonl'
resp = requests.get(hc3_url)
resp.raise_for_status()
records = []
for line in resp.text.strip().split('\n'):
    row = json.loads(line)
    for ans in row.get('human_answers', []):
        if ans.strip():
            records.append({'text': ans.strip(), 'label': 0, 'source': 'hc3_human'})
    for ans in row.get('chatgpt_answers', []):
        if ans.strip():
            records.append({'text': ans.strip(), 'label': 1, 'source': 'hc3_chatgpt'})
save_jsonl(records, DATA_DIR / 'hc3_wiki.jsonl')

# --- RAID (streaming) ---
print('\nDownloading RAID (streaming)...')
raid = load_dataset('liamdugan/raid', 'raid', split='train', streaming=True)
human_recs, ai_recs = [], []
for row in raid:
    if row.get('language', 'en') != 'en':
        continue
    text = row.get('generation', '')
    if not text or len(text.strip()) < 50:
        continue
    label = 0 if row.get('model') == 'human' else 1
    rec = {'text': text.strip(), 'label': label, 'source': 'raid'}
    if label == 0:
        human_recs.append(rec)
    else:
        ai_recs.append(rec)
    if len(human_recs) >= 1500 and len(ai_recs) >= 1500:
        break
n = min(1000, len(human_recs), len(ai_recs))
records = random.sample(human_recs, n) + random.sample(ai_recs, n)
random.shuffle(records)
save_jsonl(records, DATA_DIR / 'raid.jsonl')

# --- Summary ---
print('\n=== Dataset Summary ===')
for f in sorted(DATA_DIR.glob('*.jsonl')):
    with open(f) as fh:
        lines = fh.readlines()
    labels = [json.loads(l)['label'] for l in lines]
    n0, n1 = labels.count(0), labels.count(1)
    print(f'  {f.name:30s}  {len(lines):>6d} samples  (human={n0}, ai={n1})')

  Done.

  Saved 1684 records to hc3_wiki.jsonl



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


  Saved 2000 records to raid.jsonl

=== Dataset Summary ===
  hc3_wiki.jsonl                    1684 samples  (human=842, ai=842)
  raid.jsonl                        2000 samples  (human=1000, ai=1000)
  train.jsonl                      23707 samples  (human=9101, ai=14606)
  val.jsonl                         3589 samples  (human=1277, ai=2312)


In [4]:
# =====================================================
# CELL 3 — Utilities (inlined: chunking, metrics)
# =====================================================
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, brier_score_loss, f1_score, fbeta_score
from tqdm import tqdm

CHUNK_SIZE = 512
CHUNK_STRIDE = 256


def chunk_text_by_tokens(input_ids, attention_mask, chunk_size=512,
                         stride=256, pad_token_id=0):
    if len(input_ids) <= chunk_size:
        return [input_ids], [attention_mask]
    chunked_ids, chunked_masks = [], []
    start = 0
    while start < len(input_ids):
        end = min(start + chunk_size, len(input_ids))
        c_ids = input_ids[start:end]
        c_mask = attention_mask[start:end]
        if len(c_ids) < chunk_size:
            pad_len = chunk_size - len(c_ids)
            c_ids = c_ids + [pad_token_id] * pad_len
            c_mask = c_mask + [0] * pad_len
        chunked_ids.append(c_ids)
        chunked_masks.append(c_mask)
        start += stride
        if start + stride >= len(input_ids):
            break
    return chunked_ids, chunked_masks


def get_all_metrics(y_true, y_pred, y_prob):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)
    n = len(y_true)
    unanswered = (y_prob == 0.5)
    n_u = np.sum(unanswered)
    n_c = np.sum((y_pred == y_true) & (~unanswered))
    c_at_1 = (1/n) * (n_c + n_u * (n_c / n)) if n > 0 else 0
    mod_pred = y_pred.copy()
    mod_pred[unanswered] = 0
    return {
        'ROC-AUC': float(roc_auc_score(y_true, y_prob)),
        'Brier': float(brier_score_loss(y_true, y_prob)),
        'F1': float(f1_score(y_true, y_pred)),
        'C@1': float(c_at_1),
        'F0.5u': float(fbeta_score(y_true, mod_pred, beta=0.5)),
    }


def predict_one_chunked(model, tokenizer, text, device):
    """Predict P(AI) for a single text with chunking."""
    enc = tokenizer(text, truncation=False, return_tensors='pt', add_special_tokens=True)
    ids = enc['input_ids'][0].tolist()
    mask = enc['attention_mask'][0].tolist()
    c_ids, c_masks = chunk_text_by_tokens(
        ids, mask, chunk_size=CHUNK_SIZE, stride=CHUNK_STRIDE,
        pad_token_id=tokenizer.pad_token_id or 0,
    )
    probs = []
    with torch.no_grad():
        for ci, cm in zip(c_ids, c_masks):
            inp = {
                'input_ids': torch.tensor([ci]).to(device),
                'attention_mask': torch.tensor([cm]).to(device),
            }
            out = model(**inp)
            p = torch.softmax(out.logits.float(), dim=1)
            probs.append(p[0, 1].item())
    return float(np.mean(probs))


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]


print('Utilities loaded.')

Utilities loaded.


## Load Datasets

In [5]:
# =====================================================
# CELL 4 — Load all datasets into memory
# =====================================================
DATASETS = {
    'PAN2025': DATA_DIR / 'val.jsonl',
    'HC3': DATA_DIR / 'hc3_wiki.jsonl',
    'RAID': DATA_DIR / 'raid.jsonl',
}

dataset_cache = {}
for name, path in DATASETS.items():
    if not path.exists():
        print(f'WARNING: {name} not found at {path}')
        continue
    records = load_jsonl(str(path))
    texts = [r['text'] for r in records]
    labels = np.array([r['label'] for r in records])
    dataset_cache[name] = {'texts': texts, 'labels': labels}
    print(f'{name}: {len(texts)} samples, {labels.mean():.1%} AI')

PAN2025: 3589 samples, 64.4% AI
HC3: 1684 samples, 50.0% AI
RAID: 2000 samples, 50.0% AI


## Load Homoglyph-Augmented Model

In [7]:
# =====================================================
# CELL 5 — Load Qwen LoRA (homoglyph) from HuggingFace
# =====================================================
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from peft import PeftModel

QWEN_BASE = 'Qwen/Qwen2.5-1.5B'
QWEN_ADAPTER = 'hersheys-baklava/pan2026-qwen-homoglyph'

print(f'Loading base model: {QWEN_BASE}')
qwen_config = AutoConfig.from_pretrained(QWEN_BASE)
qwen_config.pad_token_id = qwen_config.eos_token_id
qwen_config.num_labels = 2
qwen_config.problem_type = 'single_label_classification'

qwen_base = AutoModelForSequenceClassification.from_pretrained(
    QWEN_BASE, config=qwen_config, torch_dtype=torch.bfloat16,
)

print(f'Loading LoRA adapter: {QWEN_ADAPTER}')
qwen_model = PeftModel.from_pretrained(qwen_base, QWEN_ADAPTER)
qwen_model = qwen_model.merge_and_unload()
qwen_model.to(device)
qwen_model.eval()

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_ADAPTER)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token
qwen_tokenizer.padding_side = 'left'

def qwen_predict_proba(texts):
    probs = []
    for text in tqdm(texts, desc='Qwen LoRA (Homoglyph)', leave=False):
        probs.append(predict_one_chunked(qwen_model, qwen_tokenizer, text, device))
    return np.array(probs)

# Quick test
test_prob = predict_one_chunked(qwen_model, qwen_tokenizer, 'This is a test.', device)
print(f'Qwen LoRA (Homoglyph) loaded. Test P(AI): {test_prob:.4f}')

Loading base model: Qwen/Qwen2.5-1.5B


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading LoRA adapter: hersheys-baklava/pan2026-qwen-homoglyph


adapter_config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/8.74M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/2.43k [00:00<?, ?B/s]

Qwen LoRA (Homoglyph) loaded. Test P(AI): 0.4206


## Evaluate on All Datasets

In [8]:
# =====================================================
# CELL 6 — Run predictions on all datasets
# =====================================================
all_results = {}

for ds_name, ds_data in dataset_cache.items():
    print(f'\n{"="*60}')
    print(f'Dataset: {ds_name} ({len(ds_data["texts"])} samples)')
    print(f'{"="*60}')

    probs = qwen_predict_proba(ds_data['texts'])
    y_pred = (probs >= 0.5).astype(int)
    metrics = get_all_metrics(ds_data['labels'], y_pred, probs)

    all_results[ds_name] = metrics

    print(f'  ROC-AUC: {metrics["ROC-AUC"]:.4f}')
    print(f'  Brier:   {metrics["Brier"]:.4f}')
    print(f'  F1:      {metrics["F1"]:.4f}')
    print(f'  C@1:     {metrics["C@1"]:.4f}')
    print(f'  F0.5u:   {metrics["F0.5u"]:.4f}')

print('\nAll evaluations complete.')


Dataset: PAN2025 (3589 samples)


  ROC-AUC: 0.9998
  Brier:   0.0021
  F1:      0.9978
  C@1:     0.9972
  F0.5u:   0.9989

Dataset: HC3 (1684 samples)


  ROC-AUC: 0.9984
  Brier:   0.0060
  F1:      0.9934
  C@1:     0.9935
  F0.5u:   0.9959

Dataset: RAID (2000 samples)


  ROC-AUC: 0.8082
  Brier:   0.1602
  F1:      0.8032
  C@1:     0.8284
  F0.5u:   0.8808

All evaluations complete.


## Comparison: Homoglyph vs Baseline (006)

In [9]:
# =====================================================
# CELL 7 — Compare against 006 baseline
# =====================================================

# Experiment 006 baseline results
baseline_006 = {
    'PAN2025': {'ROC-AUC': 0.9999, 'Brier': 0.0019, 'F1': 0.9981, 'C@1': 0.9975, 'F0.5u': 0.9984},
    'HC3':     {'ROC-AUC': 0.9982, 'Brier': 0.0112, 'F1': 0.9868, 'C@1': 0.9869, 'F0.5u': 0.9947},
    'RAID':    {'ROC-AUC': 0.8152, 'Brier': 0.1491, 'F1': 0.8182, 'C@1': 0.8460, 'F0.5u': 0.9176},
}

print('ROC-AUC Comparison: Homoglyph (008) vs Baseline (006)')
print('=' * 60)
print(f'{"Dataset":<10} {"006 Baseline":>12} {"008 Homoglyph":>14} {"Delta":>10}')
print('-' * 60)

ood_delta_sum = 0
ood_count = 0
for ds_name in ['PAN2025', 'HC3', 'RAID']:
    b = baseline_006[ds_name]['ROC-AUC']
    h = all_results[ds_name]['ROC-AUC']
    d = h - b
    marker = '+' if d > 0 else '' if d == 0 else ''
    print(f'{ds_name:<10} {b:>12.4f} {h:>14.4f} {d:>+10.4f}')
    if ds_name != 'PAN2025':
        ood_delta_sum += d
        ood_count += 1

avg_ood_base = (baseline_006['HC3']['ROC-AUC'] + baseline_006['RAID']['ROC-AUC']) / 2
avg_ood_homo = (all_results['HC3']['ROC-AUC'] + all_results['RAID']['ROC-AUC']) / 2
print('-' * 60)
print(f'{"Avg OOD":<10} {avg_ood_base:>12.4f} {avg_ood_homo:>14.4f} {avg_ood_homo - avg_ood_base:>+10.4f}')

ROC-AUC Comparison: Homoglyph (008) vs Baseline (006)
Dataset    006 Baseline  008 Homoglyph      Delta
------------------------------------------------------------
PAN2025          0.9999         0.9998    -0.0001
HC3              0.9982         0.9984    +0.0002
RAID             0.8152         0.8082    -0.0070
------------------------------------------------------------
Avg OOD          0.9067         0.9033    -0.0034


In [10]:
# =====================================================
# CELL 8 — Full metrics comparison table
# =====================================================
metric_names = ['ROC-AUC', 'Brier', 'F1', 'C@1', 'F0.5u']

rows = []
for ds_name in ['PAN2025', 'HC3', 'RAID']:
    for metric in metric_names:
        b = baseline_006[ds_name][metric]
        h = all_results[ds_name][metric]
        d = h - b
        rows.append({
            'Dataset': ds_name,
            'Metric': metric,
            '006 Baseline': b,
            '008 Homoglyph': h,
            'Delta': d,
        })

df_compare = pd.DataFrame(rows)
print(df_compare.to_string(index=False, float_format='%.4f'))

Dataset  Metric  006 Baseline  008 Homoglyph   Delta
PAN2025 ROC-AUC        0.9999         0.9998 -0.0001
PAN2025   Brier        0.0019         0.0021  0.0002
PAN2025      F1        0.9981         0.9978 -0.0003
PAN2025     C@1        0.9975         0.9972 -0.0003
PAN2025   F0.5u        0.9984         0.9989  0.0005
    HC3 ROC-AUC        0.9982         0.9984  0.0002
    HC3   Brier        0.0112         0.0060 -0.0052
    HC3      F1        0.9868         0.9934  0.0066
    HC3     C@1        0.9869         0.9935  0.0066
    HC3   F0.5u        0.9947         0.9959  0.0012
   RAID ROC-AUC        0.8152         0.8082 -0.0070
   RAID   Brier        0.1491         0.1602  0.0111
   RAID      F1        0.8182         0.8032 -0.0150
   RAID     C@1        0.8460         0.8284 -0.0176
   RAID   F0.5u        0.9176         0.8808 -0.0368


In [11]:
# =====================================================
# CELL 9 — Save results
# =====================================================
from datetime import datetime

results_payload = {
    'experiment': '008-homoglyph',
    'evaluated_at': datetime.now().isoformat(),
    'device': device,
    'model': 'hersheys-baklava/qwen-lora-homoglyph',
    'augmentation': {
        'h_probability': 0.05,
        'zwj_probability': 0.05,
        'aug_fraction': 0.10,
    },
    'datasets': list(dataset_cache.keys()),
    'results': all_results,
    'baseline_006': baseline_006,
}

with open('/content/homoglyph_evaluation_results.json', 'w') as f:
    json.dump(results_payload, f, indent=2)
print('Saved homoglyph_evaluation_results.json')

df_compare.to_csv('/content/homoglyph_comparison.csv', index=False)
print('Saved homoglyph_comparison.csv')

print(f'\nDone. Download from /content/ to save locally.')

Saved homoglyph_evaluation_results.json
Saved homoglyph_comparison.csv

Done. Download from /content/ to save locally.


In [13]:
# =====================================================
# CELL 10 — Upload results to HuggingFace
# =====================================================
from huggingface_hub import HfApi, login

HF_TOKEN = "hf_xxx"
login(token=HF_TOKEN)

api = HfApi()
api.upload_file(
    path_or_fileobj='/content/homoglyph_evaluation_results.json',
    path_in_repo='homoglyph_evaluation_results.json',
    repo_id='hersheys-baklava/pan2026-qwen-homoglyph',
    commit_message='Upload evaluation results for 008-homoglyph',
)
print('Results uploaded to HuggingFace.')

Results uploaded to HuggingFace.
